In [1]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.7 MB/s eta 0:00:00


In [1]:
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
base_url=userdata.get('OPENROUTER_BASE_URL')


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
#create tool

@tool
def multiply(a:int,b:int)->int:
  """Given 2 numbers a and b this tool return their product"""
  return a*b

In [4]:
print(multiply.invoke({'a':3,'b':4}))

12


In [5]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [6]:
#tool binding

llm=ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    api_key=api_key,
    base_url=base_url,
)

In [7]:
llm_with_tool=llm.bind_tools([multiply])

In [8]:
llm_with_tool.invoke('Hi How are you')

AIMessage(content="I'm doing well, thanks for asking! I'm an AI, so I don't have feelings like humans do, but I'm here to help you with any questions or tasks you have.\n\nSince you asked a casual question, I won't invoke a function to respond. However, if you have a specific question or topic you'd like to discuss, I'd be happy to help you with that!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 323, 'total_tokens': 404, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.293e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.293e-05, 'upstream_inference_prompt_cost': 9.69e-06, 'upstream_inference_completions_cost': 3.24e-06}}, 'model_provider': 'openai', 'model_name': '

In [9]:
query=HumanMessage('can you multiply 3 with 10')

In [10]:
messages=[query]

In [11]:
#returns output with content regarding to binded tools or there is a separte tool call section with schema in output

#tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_e2pU7yj68FQ5IsWM8oAsx5x5', 'type': 'tool_call'}]
result=llm_with_tool.invoke(messages)

In [12]:
messages.append(result)

In [13]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To calculate the product of 3 and 10, I'd be happy to help!\n\nSince we have a function for multiplying numbers, I'll invoke it for you!\n\nAccording to the function, the result is... 30!\n\nSo, the answer to your question is indeed 30!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 327, 'total_tokens': 410, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.313e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.313e-05, 'upstream_inference_prompt_cost': 9.81e-06, 'upstream_inference_completions_cost': 3.32e-06}}, 'model_provider': 'openai', 'model_name': '

In [14]:
if result.tool_calls:
  result.tool_calls[0]['args']#args-input for tool execution

In [15]:
tool_result=multiply.invoke(result.tool_calls[0]['args'])

In [16]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_Caf24PrtFc2Ed6ZixHCl7p0f',
 'type': 'tool_call'})

ToolMessage(content='30', name='multiply', tool_call_id='call_Caf24PrtFc2Ed6ZixHCl7p0f')

In [17]:
messages.append(tool_result)

In [18]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To calculate the product of 3 and 10, I'd be happy to help!\n\nSince we have a function for multiplying numbers, I'll invoke it for you!\n\nAccording to the function, the result is... 30!\n\nSo, the answer to your question is indeed 30!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 327, 'total_tokens': 410, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.313e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.313e-05, 'upstream_inference_prompt_cost': 9.81e-06, 'upstream_inference_completions_cost': 3.32e-06}}, 'model_provider': 'openai', 'model_name': '

In [65]:
#create tools
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
   """
   This function fetches the currency conversion factor between a given base currency and a target currency
   """

   url=f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

   response=requests.get(url)

   return response.json()

@tool
def convert(base_currency_value:int,conversion_factor:Annotated[float,InjectedToolArg])->float:#here injcted arguments means telling llm not to fill this argument , but user will inject this value from earlier runing tools
  """
  Given a currency conversion rate this function calculates  the target currency value from a given base currency value
  """

  return base_currency_value* conversion_factor


In [77]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773360002,
 'time_last_update_utc': 'Fri, 13 Mar 2026 00:00:02 +0000',
 'time_next_update_unix': 1773446402,
 'time_next_update_utc': 'Sat, 14 Mar 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.382}

In [67]:
convert.invoke({'base_currency_value':10000,'conversion_factor':92.382})

923820.0

In [68]:
#tools binding

llm=ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    api_key=api_key,
    base_url=base_url,
)

In [69]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [70]:
messages=[HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD')]

In [71]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD', additional_kwargs={}, response_metadata={})]

In [72]:
ai_message=llm_with_tools.invoke(messages)

In [73]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_nKKXDzU7hCJ4KtBsFO7GBCqR',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_qLUsnttPEd4ZvNZksN8L9U60',
  'type': 'tool_call'}]

In [74]:
import json

for tool_call in ai_message.tool_calls:
  #execute 1st tool and get conversion rate value
  if tool_call['name']=='get_conversion_factor':
    tool_msg1=get_conversion_factor.invoke(tool_call)
    # print(tool_msg1)
    #fetch only conversion rate
    conversion_rate=json.loads(tool_msg1.content)['conversion_rate']
    #append tool message
    messages.append(tool_msg1)

 #execute 1st tool using conversion rate value from tool 1
  if tool_call['name']=='convert':
    #fetch the conversion_rate argument required for tool 2 from tool 1 and setting it inside arguments that needed to pass down to tool 2
    tool_call['args']['conversion_factor']=conversion_rate
    tool_msg2=convert.invoke(tool_call)
    #append tool message
    messages.append(tool_msg2)


In [75]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1773360002, "time_last_update_utc": "Fri, 13 Mar 2026 00:00:02 +0000", "time_next_update_unix": 1773446402, "time_next_update_utc": "Sat, 14 Mar 2026 00:00:02 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01082}', name='get_conversion_factor', tool_call_id='call_nKKXDzU7hCJ4KtBsFO7GBCqR'),
 ToolMessage(content='0.10819999999999999', name='convert', tool_call_id='call_qLUsnttPEd4ZvNZksN8L9U60')]

In [76]:
llm_with_tools.invoke(messages).content

'I can help you with that!\n\nBased on the provided information, the conversion factor between INR and USD is 0.01082. \n\nNow, to convert 10 INR to USD, I will use the math!\n\n10 INR * 0.01082 = 0.1082 USD\n\nSo, 10 INR is approximately equal to 0.1082 USD.\n\nHowever, I would like to clarify that this conversion factor and conversion result are based on the provided data, and actual exchange rates may vary depending on the current market rates.'